# University Library API — Demo Notebook

This notebook walks through:
1. How account creation actually works
2. Registering a staff account
3. Logging in and getting a JWT
4. Calling several authenticated endpoints

> **Prerequisites:** The API and MySQL containers must be running (`docker compose up`).

In [ ]:
import requests
import json

BASE_URL = "http://localhost:8000"

def pretty(response):
    """Print status code + formatted JSON body."""
    print(f"HTTP {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=2, default=str))
    except Exception:
        print(response.text)

---
## ⚠️ How Account Creation Actually Works

**The short version:** `POST /auth/register` is intentionally open — no token required.

Here's the full flow that happens when you register:

```
POST /auth/register  { first_name, last_name, email, password, library_id }
         │
         ▼
  Check email uniqueness in STAFF table
         │
         ▼
  hash_password(password)  ← bcrypt, 12 rounds, never stored in plain text
         │
         ▼
  INSERT INTO STAFF (first_name, last_name, email, hashed_password, library_id)
         │
         ▼
  Returns StaffOut  ← hashed_password is NOT included in the response
```

**Important caveats:**

- `library_id` is **required** and must reference an existing row in the `LIBRARY` table. 
  This means you need at least one University → Location → Library in the DB before you can register.
- The `hashed_password` column **does not exist in the original SQL schema** — you need to run this migration once after `proj2-makeSchema.sql`:
  ```sql
  ALTER TABLE STAFF ADD COLUMN hashed_password VARCHAR(255) NOT NULL DEFAULT '';
  ```
- `/auth/register` is open by design for bootstrapping. In production you'd want to gate it behind an existing admin token.
- Passwords are hashed with **bcrypt** via `passlib`. The plain-text password never touches the database.

---
## Step 1 — Health check

In [ ]:
pretty(requests.get(f"{BASE_URL}/"))

---
## Step 2 — Bootstrap: University → Location → Library

These three endpoints are **unauthenticated** only because we haven't registered yet.
Actually — wait, they require a token too. So we have a chicken-and-egg problem:
you need a `library_id` to register, but all library endpoints need auth.

**Solution:** Seed the database directly via SQL first, then register.
Run this in your MySQL container or a migration script:

```sql
INSERT INTO UNIVERSITY (name) VALUES ('Demo University');
INSERT INTO LOCATION (name, city, suburb_or_campus, university_id) VALUES ('Main Campus', 'Austin', 'Central', 1);
INSERT INTO LIBRARY (name, location_id) VALUES ('Main Library', 1);
ALTER TABLE STAFF ADD COLUMN hashed_password VARCHAR(255) NOT NULL DEFAULT '';
```

After that, `library_id = 1` is valid and you can register below.

---
## Step 3 — Register a staff account

In [ ]:
register_payload = {
    "first_name": "Alice",
    "last_name": "Smith",
    "email": "alice@library.edu",
    "password": "supersecret123",
    "library_id": 1          # Must exist in the LIBRARY table
}

resp = requests.post(f"{BASE_URL}/auth/register", json=register_payload)
pretty(resp)

# Note: the response contains staff_id, name, email, library_id
# but NOT the password or hashed_password — those never leave the server.

---
## Step 4 — Log in and get a JWT

The `/auth/token` endpoint uses the **OAuth2 password flow**, which means the
body must be sent as `application/x-www-form-urlencoded` (not JSON).
The field names are fixed by the OAuth2 spec: `username` and `password`.

In [ ]:
login_resp = requests.post(
    f"{BASE_URL}/auth/token",
    data={                        # <-- form-encoded, not json=
        "username": "alice@library.edu",
        "password": "supersecret123",
    }
)
pretty(login_resp)

token = login_resp.json()["access_token"]
headers = {"Authorization": f"Bearer {token}"}
print("\nToken stored. All subsequent requests will use it.")

---
## Step 5 — Who am I?

In [ ]:
pretty(requests.get(f"{BASE_URL}/auth/me", headers=headers))

---
## Step 6 — Create a University, Location, and Library via the API

Now that we're authenticated, we can use the API properly instead of raw SQL.

In [ ]:
# Create a university
uni_resp = requests.post(
    f"{BASE_URL}/universities/",
    json={"name": "State University"},
    headers=headers
)
pretty(uni_resp)
university_id = uni_resp.json()["university_id"]

In [ ]:
# Create a location under that university
loc_resp = requests.post(
    f"{BASE_URL}/locations/",
    json={
        "name": "North Campus",
        "city": "Springfield",
        "suburb_or_campus": "North",
        "university_id": university_id
    },
    headers=headers
)
pretty(loc_resp)
location_id = loc_resp.json()["location_id"]

In [ ]:
# Create a library at that location
lib_resp = requests.post(
    f"{BASE_URL}/libraries/",
    json={"name": "North Campus Library", "location_id": location_id},
    headers=headers
)
pretty(lib_resp)
library_id = lib_resp.json()["library_id"]

---
## Step 7 — Add a book to the catalog

In [ ]:
# Create an author
author_resp = requests.post(
    f"{BASE_URL}/authors/",
    json={"first_name": "Jane", "last_name": "Austen"},
    headers=headers
)
pretty(author_resp)
author_id = author_resp.json()["author_id"]

In [ ]:
# Create the item title
item_resp = requests.post(
    f"{BASE_URL}/items/",
    json={
        "title": "Pride and Prejudice",
        "media_type": "Book",
        "loc_classification": "PR4034.P7",
        "publication_date": "1813-01-28",
        "is_electronic": False
    },
    headers=headers
)
pretty(item_resp)
item_id = item_resp.json()["item_id"]

In [ ]:
# Link the author to the item
pretty(requests.post(
    f"{BASE_URL}/item-authors/",
    json={"item_id": item_id, "author_id": author_id},
    headers=headers
))

In [ ]:
# Add a physical copy at our library
copy_resp = requests.post(
    f"{BASE_URL}/copies/",
    json={
        "item_id": item_id,
        "library_id": library_id,
        "status": "Available",
        "available_for_checkout": True
    },
    headers=headers
)
pretty(copy_resp)
copy_id = copy_resp.json()["copy_id"]

---
## Step 8 — Register a patron and check out the book

In [ ]:
from datetime import date, timedelta

# Create a patron (base record)
patron_resp = requests.post(
    f"{BASE_URL}/patrons/",
    json={
        "patron_type": "Student",
        "email": "bob.jones@university.edu",
        "phone": "555-1234"
    },
    headers=headers
)
pretty(patron_resp)
patron_id = patron_resp.json()["patron_id"]

In [ ]:
# Check out the book
today = date.today()
due   = today + timedelta(days=14)

checkout_resp = requests.post(
    f"{BASE_URL}/checkouts/",
    json={
        "patron_id": patron_id,
        "copy_id": copy_id,
        "checkout_date": str(today),
        "due_date": str(due)
    },
    headers=headers
)
pretty(checkout_resp)
checkout_id = checkout_resp.json()["checkout_id"]

In [ ]:
# Mark the copy as checked out
pretty(requests.patch(
    f"{BASE_URL}/copies/{copy_id}",
    json={"status": "Checked Out", "available_for_checkout": False},
    headers=headers
))

---
## Step 9 — Return the book and issue a fine

In [ ]:
# Record the return
pretty(requests.patch(
    f"{BASE_URL}/checkouts/{checkout_id}/return",
    json={"return_date": str(today)},
    headers=headers
))

In [ ]:
# Issue a fine (e.g. for a damaged page)
fine_resp = requests.post(
    f"{BASE_URL}/fines/",
    json={
        "patron_id": patron_id,
        "checkout_id": checkout_id,
        "amount": "5.00",
        "reason": "Damaged cover",
        "assessed_date": str(today)
    },
    headers=headers
)
pretty(fine_resp)
fine_id = fine_resp.json()["fine_id"]

In [ ]:
# Mark the fine as paid
pretty(requests.patch(
    f"{BASE_URL}/fines/{fine_id}",
    json={"paid_status": True},
    headers=headers
))

---
## Step 10 — Query and filter

In [ ]:
# All available copies in our library
pretty(requests.get(
    f"{BASE_URL}/copies/",
    params={"library_id": library_id, "status": "Available"},
    headers=headers
))

In [ ]:
# All checkouts for our patron
pretty(requests.get(
    f"{BASE_URL}/checkouts/",
    params={"patron_id": patron_id},
    headers=headers
))

In [ ]:
# All unpaid fines
pretty(requests.get(
    f"{BASE_URL}/fines/",
    params={"paid_status": False},
    headers=headers
))

In [ ]:
# All libraries at our location
pretty(requests.get(
    f"{BASE_URL}/libraries/",
    params={"location_id": location_id},
    headers=headers
))

---
## What to explore next

| Endpoint group | Things to try |
|---|---|
| `/holds/` | Place a hold on an item, update its status to `Ready` |
| `/rooms/` + `/reservations/` | Create a study room, reserve it for a patron |
| `/inter-library-loans/` | Transfer a copy between two libraries |
| `/staff/` + `/roles/` | Create roles like `Librarian`, assign to staff |
| `/checkouts/?overdue_only=true` | Find all currently overdue items |

Full interactive docs are available at **http://localhost:8000/docs** (Swagger UI).